# Lab 2 · Exercise A — Anatomy of a fit
Same data and model as Lab 1, Exercise C. This time we call **MIGRAD, HESSE and MINOS** ourselves with `RooMinimizer`.

In [ ]:
import ROOT
# interactive drawing in the notebook (needed on Binder)
%jsroot on
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

In [ ]:
m = ROOT.RooRealVar("m", "mass", 5080, 5540, "MeV/c^2")
data = ROOT.RooDataSet.read("../../examples/lhcb-b0-ppbar/data/rarest_b0_decay.dat", ROOT.RooArgSet(m), "D")

mean  = ROOT.RooRealVar("mean", "mean", 5280, 5200, 5350)
sigma = ROOT.RooRealVar("sigma", "sigma", 15, 5, 50)
slope = ROOT.RooRealVar("slope", "slope", -0.002, -0.02, 0)
sig   = ROOT.RooGaussian("sig", "sig", m, mean, sigma)
bkg   = ROOT.RooExponential("bkg", "bkg", m, slope)
nsig  = ROOT.RooRealVar("nsig", "signal yield", 30, 0, 166)
nbkg  = ROOT.RooRealVar("nbkg", "background yield", 130, 0, 500)
model = ROOT.RooAddPdf("model", "sig+bkg", ROOT.RooArgList(sig, bkg), ROOT.RooArgList(nsig, nbkg))
print(data.numEntries(), "events")

#### Visualize the Data

In [ ]:
# TODO: Before diving into the minimizer, let's look at the data 
# using the tools you learned in Lab 1.

## A1 · NLL, MIGRAD, HESSE

`fitTo()` is just a wrapper. Let's do it explicitly:
1. Create the extended NLL, 
2. initialize a minimizer
3. run Minuit2 MIGRAD, 
4. then HESSE. 

Save the result and read the status, the EDM and `covQual`.

In [ ]:
# 1. Build NLL
nll   = None   # TODO: model.createNLL(data, ROOT.RooFit.Extended(True))
# 2. Initialize Minimizer
minim = None   # TODO: ROOT.RooMinimizer(nll)

# 3. Run MIGRAD
# TODO: minim.setPrintLevel(-1); minim.minimize("Minuit2", "Migrad"); minim.hesse()
res = None     # TODO: res = minim.save()

# 4. Run HESSE
err_hesse = nsig.getError()      # keep it: MINOS will update the error object later
print("status =", res.status(), " covQual =", res.covQual(), " EDM =", res.edm())
print("nsig = %.2f +- %.2f (HESSE)" % (nsig.getVal(), err_hesse))

# Notice how nsig and nbkg have updated automatically!


In [ ]:
check("minimiser objects exist", nll is not None and minim is not None and res is not None)
check("fit converged (status 0)", res.status() == 0)
check("full covariance matrix (covQual 3)", res.covQual() == 3)

#### Visualize data and model

In [ ]:
# TODO: let's plot the data, the model, and the background component
# using the tools you learned in Lab 1.

## A2 · MINOS
Compute MINOS errors for `nsig` only. Compare with HESSE.

In [ ]:
# Run MINOS on nsig
# TODO: minim.minos(ROOT.RooArgSet(nsig))
lo = nsig.getErrorLo()
hi = nsig.getErrorHi()
print("MINOS: nsig = %.2f  %+.2f / %+.2f" % (nsig.getVal(), lo, hi))
print("HESSE: nsig = %.2f +- %.2f" % (nsig.getVal(), err_hesse))

In [ ]:
check("MINOS lower error is negative and upper positive", lo < 0 < hi)

## A3 · Correlations
Print the correlation matrix of the fit. Which pair is most correlated?

In [ ]:
# TODO: print the correlation matrix of res;  res.correlationMatrix().Print()
# TODO: print the correlation between "nsig" and "nbkg":  res.correlation("nsig", "nbkg")

## A4 · NLL scan, profile, and the HESSE parabola

We want to visualize the likelihood for `nsig`. But what about the other parameter (`nbkg`)?
- **Case 1 (Slice)**: We freeze `nbkg` at its best-fit value. This ignores its uncertainty.
- **Case 2 (Profile)**: We let `nbkg` re-fit (re-minimize) at each step of `nsig`. This is the correct frequentist approach.

*Note: We restrict the range of `nsig` to [10, 60] to avoid numerical instabilities and weird bumps in the NLL at high values.*

Plot three curves of $-\ln L$ against `nsig`, shifted to zero at the minimum:
the **slice** (other parameters frozen), the **profile** (other parameters re-fitted), and the **parabola** $0.5\,((x-\hat n)/\sigma_{\rm HESSE})^2$.

In [ ]:
frame = nsig.frame(ROOT.RooFit.Range(10, 75), ROOT.RooFit.Title("-lnL vs nsig"))
frame.SetMaximum(4)
frame.SetMinimum(0)
# TODO: slice:   nll.plotOn(frame, ROOT.RooFit.ShiftToZero())
# TODO: profile: prof = nll.createProfile(ROOT.RooArgSet(nsig));  prof.plotOn(frame, ROOT.RooFit.LineColor(ROOT.kRed))

# Draw a parabola with the same minimum and curvature as the NLL
parab = ROOT.TF1("parab", "0.5*((x-[0])/[1])**2", 10, 75)
parab.SetParameters(nsig.getVal(), err_hesse)
parab.SetLineStyle(2)

# Draw horizontal line at 0.5 (1 sigma for NLL, or 1.0 for -2lnL)
line = ROOT.TLine(10, 0.5, 60, 0.5)
line.SetLineColor(ROOT.kGreen)
line.SetLineStyle(ROOT.kDotted)

cs = ROOT.TCanvas("cs", "cs", 800, 500)
frame.Draw()
parab.Draw("same")
line.Draw("same")
cs.Draw()

## A5: MINOS and Contours

`MINOS` finds asymmetric errors by scanning the profile likelihood. 
We can also visualize the joint uncertainty of two parameters using a **Contour**.

*Important RooFit Tip*: The `contour` method relies on the covariance matrix. Always call `hesse()` immediately before `contour()` to ensure the minimizer is in a clean, accurate state. Otherwise, you might see stray points or distorted ellipses.

Draw the 1σ and 2σ contours of (`nsig`, `nbkg`) with `minim.contour(...)`.

In [ ]:
# Ensure clean state to avoid stray dots in the contour!
minim.hesse()

# TODO: cont = minim.contour(nsig, nbkg, 1, 2)   # returns a RooPlot
# TODO: draw it on a canvas

## Questions
Write your answer under each question, in 2–3 lines, and quote the numbers or plots of the steps indicated.
1. **(A1, A2)** Compare the HESSE error and the two MINOS errors on `nsig`. Are they equal? If not, which MINOS error is longer (lower or upper), and what does it tell you about the shape of the likelihood?
2. **(A3)** Which pair of parameters has the largest correlation coefficient, in absolute value? Is it positive or negative? Explain with the model: if one parameter goes up, what must the other one do to keep the fit good?
3. **(A4)** In which range of `nsig` do the three curves (NLL scan, profile, HESSE parabola) differ the most, and where do they agree? What does this say about the likelihood, and which error would you quote?